In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk
from nltk.corpus import stopwords

In [46]:
data = pd.read_csv("data/amazon_reviews_sentiment.csv")

In [47]:
data.head()

,content,label
0,I was disappointed with this book. I had just ...,0
1,"This film is cute, but lacks the cleverness an...",0
2,Book arrived with crumpled front cover. It was...,0
3,"Like Michigan's The Luv'd Ones, She toiled lar...",1
4,"After sparse usasge, (a couple minutes every f...",0


In [48]:
data.isnull().sum()

content    0
label      0
dtype: int64

In [49]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   content  100000 non-null  object
 1   label    100000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.5+ MB


In [50]:
data["label"].value_counts()

label
0    50029
1    49971
Name: count, dtype: int64

In [51]:
nltk.download("stopwords")
en_stopwords =  set(stopwords.words("english"))

en_stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tyill\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [52]:
len(en_stopwords)

198

In [53]:
sentiment_preserving_words = {
    'not', 'no', 'nor', 'neither', 'never', 'none', 'nobody', 'nothing', 
    'without', 'hardly', 'scarcely', 'barely', "don't", "can't", "wasn't", 
    "isn't", "aren't", "won't", "couldn't", "shouldn't",
    'but', 'however', 'although', 'yet', 'still', 'even', 'instead', 
    'despite', 'nevertheless', 'nonetheless', 'whereas',
    'very', 'most', 'more', 'too', 'extremely', 'enough', 'less', 
    'quite', 'just', 'only', 'really', 'so', 'totally', 'absolutely', 
    'highly', 'completely', 'rather', 'bit', 'slightly', 'much',
    'against', 'than', 'if', 'unless', 'whether', 'while', 'whereas',
    'worth', 'expensive', 'cheap', 'value', 'waste' 
}

In [54]:
en_stopwords = en_stopwords - sentiment_preserving_words
len(en_stopwords)

176

In [55]:
def clean_text(text):
   if not isinstance(text, str):
      return ""
    
   text = text.lower()

   contractions = {
        "n't": " not", " 's": " is", " 're": " are", " 've": " have", 
        " 'd": " would", " 'll": " will", " 'm": " am", "can't": "cannot"
   }
   for contraction, replacement in contractions.items():
      text = text.replace(contraction, replacement)

   text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
   text = re.sub(r'<.*?>', '', text)
   text = re.sub(r'@\w+', '', text)

   text = re.sub(r'\d+', '', text)
   text = text.translate(str.maketrans('', '', string.punctuation))

   words = text.split()
   cleaned_words = [w for w in words if w not in en_stopwords]

   return " ".join(cleaned_words).strip()

In [56]:
data['clean_text'] = data['content'].apply(clean_text)

In [57]:
data[["content", "clean_text"]]

,content,clean_text
0,I was disappointed with this book. I had just ...,disappointed book just finished moonlight love...
1,"This film is cute, but lacks the cleverness an...",film cute but lacks cleverness comedy comedies...
2,Book arrived with crumpled front cover. It was...,book arrived crumpled front cover not possible...
3,"Like Michigan's The Luv'd Ones, She toiled lar...",like michigans luvd ones toiled largely unnoti...
4,"After sparse usasge, (a couple minutes every f...",sparse usasge couple minutes every months mach...
...,...,...
99995,I play her self titled 2nd album over and over...,play self titled nd album never get tired good...
99996,"I bought to amazon a ""new SCJP book""...it is i...",bought amazon new scjp bookit very bad conditi...
99997,I'm an avid reader of vampire novels including...,im avid reader vampire novels including anne r...
99998,THIS IS THE QUINTESENTIAL REFERENCE FOR ALL ER...,quintesential reference erector collectors wel...


In [58]:
data = data.drop(columns="content", axis=1)
data.head()

,label,clean_text
0,0,disappointed book just finished moonlight love...
1,0,film cute but lacks cleverness comedy comedies...
2,0,book arrived crumpled front cover not possible...
3,1,like michigans luvd ones toiled largely unnoti...
4,0,sparse usasge couple minutes every months mach...


In [59]:
data.to_csv("clean_data/temiz_ingilizce_data.csv", index=False)
print("İngilizce Veri Başarıyla Kaydedildi..")

İngilizce Veri Başarıyla Kaydedildi..
